In [46]:
# python-dotenv reads your .env file and loads the key as an environment variable
# This keeps your key out of the code entirely
from dotenv import load_dotenv
import os

load_dotenv()

# Fetch the key from the environment, not hardcoded in the file
api_key = os.getenv('CFBD_API_KEY')

In [54]:
# Import the cfbd library, which is a Python wrapper for the College Football Data API
# This gives us pre-built functions to pull roster, game, and recruiting data
import cfbd

# Import pandas, the standard Python library for working with structured data
# We'll use it to organize API responses into tables (DataFrames) we can analyze
import pandas as pd

# Import dotenv to load our API key from the .env file securely
# Import os to access environment variables
from dotenv import load_dotenv
import os

# Load the .env file so our API key is available as an environment variable
load_dotenv()

# Create a configuration object that holds our API credentials
configuration = cfbd.Configuration()

# Pull the API key from the environment variable, not hardcoded in the file
configuration.api_key['Authorization'] = os.getenv('CFBD_API_KEY')

# Tell the API to prefix our key with 'Bearer' when sending requests
configuration.api_key_prefix['Authorization'] = 'Bearer'

# Create the API client using our configuration
api_client = cfbd.ApiClient(configuration)

In [55]:
# Test the API key directly using the requests library
# This bypasses the cfbd wrapper and hits the API raw
# so we can confirm the key works independently
import requests

# Replace placeholder with your actual key here as well
headers = {'Authorization': 'Bearer ' + api_key}

# Simple test endpoint that returns basic team info
response = requests.get('https://api.collegefootballdata.com/teams', headers=headers)

# Print the status code — 200 means success, 401 means key is still failing
print(response.status_code)
print(response.text[:500])

200
[{"id":2000,"school":"Abilene Christian","mascot":"Wildcats","abbreviation":"ACU","alternateNames":["ACU","Abilene Chrstn"],"conference":"UAC","division":null,"classification":"fcs","color":"#592d82","alternateColor":"#b1b3b3","logos":["http://a.espncdn.com/i/teamlogos/ncaa/500/2000.png","http://a.espncdn.com/i/teamlogos/ncaa/500-dark/2000.png"],"twitter":"@ACUFootball","location":{"id":5440,"name":"Wildcat Stadium (TX)","city":"Abilene","state":"TX","zip":"79601","countryCode":"US","timezone":"


In [56]:
import requests
import os
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

# Set up headers once, reuse for every API call
headers = {'Authorization': f'Bearer {os.getenv("CFBD_API_KEY")}'}

# Pull Notre Dame 2024 roster directly from the API
response = requests.get(
    'https://api.collegefootballdata.com/roster',
    headers=headers,
    params={'team': 'Notre Dame', 'year': 2024}
)

# Convert the response to a DataFrame
nd_roster = pd.DataFrame(response.json())

nd_roster.head()

,id,firstName,lastName,team,weight,height,jersey,year,position,homeCity,homeState,homeCountry,homeLatitude,homeLongitude,homeCountyFIPS,recruitIds
0,532607,Eric,Goins,Notre Dame,207,74,90.0,4,PK,Herndon,VA,USA,38.969532,-77.385948,51059,[]
1,4426420,Devyn,Ford,Notre Dame,200,71,22.0,4,RB,Stafford,VA,USA,38.422268,-77.408407,51179,[113663]
2,4426992,Howard,Cross III,Notre Dame,288,73,56.0,4,DL,Paramus,NJ,USA,40.945291,-74.073590,34003,[122604]
3,4427427,Rod,Heard II,Notre Dame,195,71,2.0,4,S,Farmington,MI,USA,42.464480,-83.376322,27037,[123182]
4,4427716,Jack,Kiser,Notre Dame,231,74,24.0,4,LB,Royal Center,IN,USA,40.864485,-86.499727,18017,[122685]


In [57]:
# Check how many players are in the roster
print(f'Total players: {len(nd_roster)}')

# See all available columns so we know what data we have
print(f'\nColumns: {nd_roster.columns.tolist()}')

# Check for missing values in each column
print(f'\nMissing values:\n{nd_roster.isnull().sum()}')

Total players: 121

Columns: ['id', 'firstName', 'lastName', 'team', 'weight', 'height', 'jersey', 'year', 'position', 'homeCity', 'homeState', 'homeCountry', 'homeLatitude', 'homeLongitude', 'homeCountyFIPS', 'recruitIds']

Missing values:
id                 0
firstName          0
lastName           0
team               0
weight             0
height             0
jersey             1
year               0
position           0
homeCity           0
homeState          1
homeCountry        0
homeLatitude      10
homeLongitude     10
homeCountyFIPS    10
recruitIds         0
dtype: int64


In [58]:
# See how many players we have at each class year
# This tells us if the roster data is complete across all classes
print(nd_roster['year'].value_counts().sort_index())

# Also check position distribution
print(f'\nPositions:\n{nd_roster["position"].value_counts()}')

year
1    16
2    31
3    26
4    48
Name: count, dtype: int64

Positions:
position
DL    20
OL    19
WR    17
LB    12
TE    10
CB    10
S      9
RB     8
PK     5
QB     5
LS     3
P      2
DB     1
Name: count, dtype: int64


In [59]:
# Document data quality observations for this endpoint
print("""
DATA QUALITY NOTES - Notre Dame 2024 Roster
--------------------------------------------
1. Total players (121) is below expected roster size (130-140)
   - Some players may be missing from API data
   
2. Year 4 count (48) is unusually high
   - API likely groups grad students and 5th years into year 4
   - This will affect attrition window calculations in V1
   
3. Missing values are minimal and non-critical for V1
   - Geographic fields (10 missing) not needed for depth analysis
""")


DATA QUALITY NOTES - Notre Dame 2024 Roster
--------------------------------------------
1. Total players (121) is below expected roster size (130-140)
   - Some players may be missing from API data

2. Year 4 count (48) is unusually high
   - API likely groups grad students and 5th years into year 4
   - This will affect attrition window calculations in V1

3. Missing values are minimal and non-critical for V1
   - Geographic fields (10 missing) not needed for depth analysis



In [61]:
# Pull Ohio State 2024 roster using the same endpoint
response_osu = requests.get(
    'https://api.collegefootballdata.com/roster',
    headers=headers,
    params={'team': 'Ohio State', 'year': 2024}
)

# Convert to DataFrame
osu_roster = pd.DataFrame(response_osu.json())

# Pull Indiana 2024 roster
response_ind = requests.get(
    'https://api.collegefootballdata.com/roster',
    headers=headers,
    params={'team': 'Indiana', 'year': 2024}
)

# Convert to DataFrame
ind_roster = pd.DataFrame(response_ind.json())

# Compare all three rosters side by side
print(f'Notre Dame - Total players: {len(nd_roster)}')
print(f'Ohio State - Total players: {len(osu_roster)}')
print(f'Indiana    - Total players: {len(ind_roster)}')

print(f'\nOhio State year distribution:\n{osu_roster["year"].value_counts().sort_index()}')
print(f'\nIndiana year distribution:\n{ind_roster["year"].value_counts().sort_index()}')

Notre Dame - Total players: 121
Ohio State - Total players: 123
Indiana    - Total players: 116

Ohio State year distribution:
year
1     4
2    31
3    30
4    58
Name: count, dtype: int64

Indiana year distribution:
year
1    20
2    21
3    26
4    49
Name: count, dtype: int64


In [62]:
# Pull Notre Dame recruiting data to see if class year data is more reliable there
response_recruiting = requests.get(
    'https://api.collegefootballdata.com/recruiting/players',
    headers=headers,
    params={'team': 'Notre Dame', 'year': 2024}
)

recruiting_data = pd.DataFrame(response_recruiting.json())

print(f'Total recruits: {len(recruiting_data)}')
print(f'\nColumns: {recruiting_data.columns.tolist()}')
print(f'\nFirst 5 rows:\n{recruiting_data.head()}')

Total recruits: 22

Columns: ['id', 'athleteId', 'recruitType', 'year', 'ranking', 'name', 'school', 'committedTo', 'position', 'height', 'weight', 'stars', 'rating', 'city', 'stateProvince', 'country', 'hometownInfo']

First 5 rows:
       id athleteId recruitType  year  ranking                   name  \
0  108739   5079748  HighSchool  2024       41  Kyngstonn Viliamu-Asa   
1  108744   5079559  HighSchool  2024       46         Guerby Lambert   
2  108745   5079645  HighSchool  2024       47           Cam Williams   
3  108766   5079369  HighSchool  2024       68                CJ Carr   
4  108771   5112630  HighSchool  2024       73            Bryce Young   

                school committedTo position  height  weight  stars  rating  \
0       St. John Bosco  Notre Dame       LB    74.0     233      4  0.9810   
1    Catholic Memorial  Notre Dame       OT    78.0     280      4  0.9785   
2       Glenbard South  Notre Dame       WR    74.0     188      4  0.9783   
3              

In [63]:
# Check what recruitIds actually look like in the roster data
# We need to confirm they match athleteId from the recruiting endpoint
print(nd_roster['recruitIds'].head(10))

0          []
1    [113663]
2    [122604]
3    [123182]
4    [122685]
5    [124004]
6    [123158]
7    [118527]
8    [118647]
9          []
Name: recruitIds, dtype: object


In [64]:
# Count how many players have no recruit ID
no_recruit_id = nd_roster[nd_roster['recruitIds'].apply(lambda x: len(x) == 0)]
print(f'Players with no recruitId: {len(no_recruit_id)}')
print(f'Players with recruitId: {len(nd_roster) - len(no_recruit_id)}')

Players with no recruitId: 42
Players with recruitId: 79


In [65]:
# See what positions the players with no recruitId play
# This tells us if the gap is concentrated in certain position groups
print(no_recruit_id['position'].value_counts())

position
WR    6
DL    5
CB    5
PK    4
TE    4
LB    4
S     3
OL    3
QB    2
P     2
RB    2
LS    2
Name: count, dtype: int64


In [66]:
print("""
V1 DATA VERDICT - CFBD API
---------------------------
SUFFICIENT FOR:
- Position group headcount analysis
- Depth gap identification by position
- Recruit discovery by position (recruiting endpoint)

NOT SUFFICIENT FOR:
- Precise attrition windows (year field unreliable)
- Full recruiting profile joins (35% of roster unlinked)

DECISION: Proceed with V1 scoped to position group depth analysis
""")


V1 DATA VERDICT - CFBD API
---------------------------
SUFFICIENT FOR:
- Position group headcount analysis
- Depth gap identification by position
- Recruit discovery by position (recruiting endpoint)

NOT SUFFICIENT FOR:
- Precise attrition windows (year field unreliable)
- Full recruiting profile joins (35% of roster unlinked)

DECISION: Proceed with V1 scoped to position group depth analysis

